[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/00_p01_tensor_basics_solution.ipynb)

# 📖 Solution: Tensor Basics

Reference solutions for every exercise in `00_p01_tensor_basics.ipynb`. Try the template first!

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

## 1. Creating tensors

| You want | Use |
|----------|-----|
| From Python data | `torch.tensor([[1., 2.], [3., 4.]])` |
| A constant | `torch.zeros(2, 3)`, `torch.ones(2, 3)`, `torch.full((2, 3), 7.0)` |
| A range | `torch.arange(5)`, `torch.arange(0, 10, 2)`, `torch.linspace(0, 1, 5)` |
| Random | `torch.randn(2, 3)` (normal), `torch.rand(2, 3)` (uniform) |
| Same shape/dtype/device as `x` | `torch.zeros_like(x)`, `torch.full_like(x, 7.0)`, `torch.rand_like(x)` |
| A **different** shape, same dtype/device as `x` | `x.new_zeros(4, 1)`, `x.new_full((4, 1), 7.0)` |

In [ ]:
x = torch.randn(2, 3, dtype=torch.float64)
print(torch.zeros_like(x).dtype)          # float64: *_like copies dtype (and device)
print(x.new_full((4, 1), -1.0))           # different shape, still float64
print(torch.arange(5), torch.linspace(0, 1, 5))

## 2. Dtypes

- Python floats become `float32`; Python ints become `int64`.
- **Comparisons return `bool`**: `x > 0` is a mask, not numbers. Convert with `.float()` or `.to(x.dtype)`.
- `.float()` always gives `float32`. `.to(x.dtype)` **keeps** whatever precision `x` had. Prefer it when matching another tensor.
- Casting an out-of-range value to a small integer type gives **garbage** silently (it typically wraps around; the exact result is platform-dependent). Always clamp to the valid range *before* casting.

In [ ]:
print(torch.tensor([1, 2]).dtype, torch.tensor([1.0, 2.0]).dtype)   # int64 float32

x = torch.tensor([-1.0, 0.5, 2.0], dtype=torch.float64)
mask = x > 0
print(mask, mask.dtype)                  # bool
print(mask.float().dtype, mask.to(x.dtype).dtype)   # float32 vs float64

big = torch.tensor([200.0, -300.0])
print("cast directly:", big.to(torch.int8))                    # garbage, e.g. [-56, -44]; varies by platform
print("clamp first:  ", big.clamp(-128, 127).to(torch.int8))   # [127, -128]

## 3. Shape basics

`.shape` is a tuple of plain Python ints, so you can unpack it and do arithmetic with it.

In [ ]:
w = torch.randn(64, 32, 3, 3)            # a conv weight: (C_out, C_in, kH, kW)
C_out, C_in, kH, kW = w.shape
print(w.ndim, w.numel(), C_in * kH * kW, w[0].numel())
print(type(w.shape[1]))                  # <class 'int'>

## 4. Devices

Every tensor lives on a device (`cpu`, `cuda:0`, `mps`, …). Operations between tensors on **different** devices raise an error.
When your function creates a new tensor, put it on the **same device as its inputs**. Pass `device=x.device`, or use `*_like` / `x.new_*`, which do this automatically.
TorchCode's judge runs on CPU, but code that hard-codes `torch.zeros(...)` breaks the moment it runs on a GPU, and interviewers notice.

In [ ]:
x = torch.randn(3, 4)
good = torch.zeros(3, 1, device=x.device, dtype=x.dtype)   # explicit
also_good = x.new_zeros(3, 1)                              # inherits device + dtype
print(x.device, good.device, also_good.device)

## 5. Python numbers vs tensors

- Reductions return **0-d tensors**: `x.sum()` is `tensor(3.)`, not `3.0`.
- `.item()` turns a one-element tensor into a Python number (e.g. for logging or returning a float).
- Python's `math` functions work on Python numbers; `torch` functions work on tensors. `math.sqrt(2.0)` as a constant inside tensor math is fine.

In [ ]:
import math
s = torch.tensor([1.0, 2.0]).sum()
print(s, s.ndim, s.item(), type(s.item()))
print(torch.tensor([1.0, 4.0]) / math.sqrt(2.0))

## 6. In-place operations

Methods ending in `_` (`add_`, `mul_`, `zero_`, `normal_`, `clamp_`) modify the tensor **in place** and return it.
Plain assignment such as `x = x * 2` creates a **new** tensor and rebinds the local name. The caller's tensor is unchanged.
In-place ops on a **view** (like a slice) modify the original tensor too.

In [ ]:
def double_rebind(x):
    x = x * 2          # new tensor; caller unaffected
    return x

def double_inplace(x):
    return x.mul_(2)   # modifies caller's tensor

t = torch.ones(3)
double_rebind(t);  print("after rebind: ", t)
double_inplace(t); print("after in-place:", t)

m = torch.zeros(2, 3)
m[0].add_(1)                             # m[0] is a view into m
print(m)

## 7. Element-wise math (reference)

These functions work element by element, keep the shape, and support autograd. Problems usually give you the formula; this table gives you the name.

| Math | PyTorch | Used in |
|------|---------|---------|
| $e^x$, $\ln x$ | `torch.exp`, `torch.log` (`torch.log1p` for $\ln(1+x)$, see P9) | softmax, losses |
| $\sin$, $\cos$ | `torch.sin`, `torch.cos` | #24 RoPE |
| $\mathrm{erf}$ | `torch.erf` | #19 GELU (exact form) |
| $\sqrt{x}$, $x^p$ | `torch.sqrt`, `x ** p` | norms, init |
| $\max(x, 0)$, $\max(a, b)$ | `torch.clamp(x, min=0)`, `torch.maximum(a, b)` | ReLU, quantization |
| $\sigma(x)$, $\tanh x$ | `torch.sigmoid`, `torch.tanh` | gates |
| SiLU $x\,\sigma(x)$ | `F.silu` | #15 SwiGLU |
| ELU | `F.elu` | #12 linear-attention feature map |
| GELU | `F.gelu`, or the module `nn.GELU()` | #13 GPT-2 block |

`torch.nn.functional` (usually imported as `F`) holds plain **functions**. The `nn.X` versions (`nn.GELU()`, `nn.ReLU()`) are **modules** you can put inside `nn.Sequential` (P8). Same math either way.

In [ ]:
import torch.nn.functional as F
x = torch.linspace(-2, 2, 5)
print(torch.erf(x))
print(F.silu(x), x * torch.sigmoid(x))          # same values
print(F.gelu(x), torch.nn.GELU()(x))            # function vs module
print(torch.sin(x) ** 2 + torch.cos(x) ** 2)    # all ones

---
# ✏️ Exercises

Implement the five functions below. Each one has a debug cell you can use before you submit.

### Exercise 1: `positive_mask(x)`

Return a tensor that is `1` where `x > 0` and `0` elsewhere.

- `x`: any shape, any floating dtype
- returns: same shape **and same dtype** as `x`

*ReLU is just `x * positive_mask(x)` (#01).*

In [ ]:
# ✅ SOLUTION

def positive_mask(x: torch.Tensor) -> torch.Tensor:
    return (x > 0).to(x.dtype)

In [ ]:
x = torch.tensor([-2.0, 0.0, 3.0], dtype=torch.float64)
m = positive_mask(x)
print(m, m.dtype)     # expect tensor([0., 0., 1.], dtype=torch.float64)

### Exercise 2: `running_max_init(scores)`

Flash Attention keeps a running row-maximum that starts at `-inf`. Create it.

- `scores`: `(B, S, D)`
- returns: `(B, S, 1)` filled with `-inf`, with the **same dtype and device** as `scores`

In [ ]:
# ✅ SOLUTION

def running_max_init(scores: torch.Tensor) -> torch.Tensor:
    B, S, _ = scores.shape
    return scores.new_full((B, S, 1), float("-inf"))

In [ ]:
scores = torch.randn(2, 4, 8, dtype=torch.float64)
m = running_max_init(scores)
print(m.shape, m.dtype, m.device)   # expect torch.Size([2, 4, 1]) torch.float64 cpu
print(m[0, :, 0])                   # expect all -inf

### Exercise 3: `scale_inplace_(x, s)`

Multiply `x` by the number `s` **in place** and return `x` itself (the same tensor object, not a copy).

- `x`: any shape
- `s`: Python float
- returns: `x`

*This is how BatchNorm updates running stats (#07), Kaiming init fills weights (#20), and gradient clipping rescales `.grad` (#21).*

In [ ]:
# ✅ SOLUTION

def scale_inplace_(x: torch.Tensor, s: float) -> torch.Tensor:
    return x.mul_(s)

In [ ]:
t = torch.ones(2, 3)
out = scale_inplace_(t, 0.5)
print(t)              # expect all 0.5: the caller's tensor changed
print(out is t)       # expect True

### Exercise 4: `quantize_int8(w, scale)`

Quantize float weights to 8-bit integers: divide by `scale`, round to the nearest integer, clamp to `[-128, 127]`, and cast to `torch.int8`.

- `w`: float tensor, any shape
- `scale`: Python float **or** a tensor that broadcasts against `w` (e.g. per-row `(N, 1)`)
- returns: `torch.int8` tensor, same shape as `w`

*The order matters. See lesson §2 for what happens if you cast before clamping. This is the core of #36.*

In [ ]:
# ✅ SOLUTION

def quantize_int8(w: torch.Tensor, scale) -> torch.Tensor:
    return torch.round(w / scale).clamp(-128, 127).to(torch.int8)

In [ ]:
w = torch.tensor([0.4, -1.2, 3.0, 500.0, -500.0])
print(quantize_int8(w, 0.1))
# expect tensor([   4,  -12,   30,  127, -128], dtype=torch.int8)

### Exercise 5: `fan_in_out(weight)`

Compute the fan-in and fan-out used by weight initializers.

- `weight`: a `Linear` weight `(out_features, in_features)` **or** a conv weight `(C_out, C_in, kH, kW)`
- returns: a tuple of two **Python ints** `(fan_in, fan_out)`
  - Linear: `fan_in = in_features`, `fan_out = out_features`
  - Conv: `fan_in = C_in * kH * kW`, `fan_out = C_out * kH * kW`

*#20 Kaiming Init uses `std = sqrt(2 / fan_in)`.*

In [ ]:
# ✅ SOLUTION

def fan_in_out(weight: torch.Tensor) -> tuple[int, int]:
    receptive_field = weight[0][0].numel()   # 1 for Linear, kH * kW for conv
    return weight.shape[1] * receptive_field, weight.shape[0] * receptive_field

In [ ]:
print(fan_in_out(torch.empty(64, 32)))          # expect (32, 64)
print(fan_in_out(torch.empty(16, 3, 5, 5)))     # expect (75, 400)

<!-- Generated from scripts/prereqs/content/p01_tensor_basics.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("p01_tensor_basics")

---
## 🔓 What this unlocks

With **Tensor Basics** in your toolkit, you're ready for these problems:

| # | Problem | Difficulty | What you'll reuse | Links |
|:---:|---|:---:|---|---|
| #1 | ReLU | 🟢 Easy | `positive_mask`: ReLU is `x` times a bool mask converted to `x.dtype` | [Open](01_relu.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/01_relu.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/01_relu.ipynb" target="_blank">Colab</a> |
| #7 | BatchNorm | 🟡 Medium | `scale_inplace_`: running stats are updated in place with `mul_` / `add_` | [Open](07_batchnorm.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb" target="_blank">Colab</a> |
| #12 | Linear Attention | 🔴 Hard | The ELU feature map `F.elu(x) + 1` (lesson §7) | [Open](12_linear_attention.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb" target="_blank">Colab</a> |
| #13 | GPT-2 Block | 🔴 Hard | `nn.GELU()` as a module inside the MLP (lesson §7) | [Open](13_gpt2_block.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">Colab</a> |
| #15 | SwiGLU MLP | 🟡 Medium | `F.silu` for the gate (lesson §7) | [Open](15_mlp.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/15_mlp.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/15_mlp.ipynb" target="_blank">Colab</a> |
| #19 | GELU | 🟢 Easy | `torch.erf` and Python constants like `math.sqrt(2.0)` in tensor math (lessons §5 and §7) | [Open](19_gelu.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/19_gelu.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/19_gelu.ipynb" target="_blank">Colab</a> |
| #24 | RoPE | 🔴 Hard | `torch.sin` / `torch.cos` of the angle table (lesson §7) | [Open](24_rope.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/24_rope.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb" target="_blank">Colab</a> |
| #20 | Kaiming Init | 🟢 Easy | `fan_in_out` for the std, and in-place `normal_` (`scale_inplace_`) | [Open](20_weight_init.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb" target="_blank">Colab</a> |
| #21 | Gradient Clipping | 🟢 Easy | `scale_inplace_` on `p.grad`, and returning the norm as a Python float with `.item()` | [Open](21_gradient_clipping.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb" target="_blank">Colab</a> |
| #25 | Flash Attention | 🔴 Hard | `running_max_init`: `-inf` running max and zero accumulators with the right dtype and device | [Open](25_flash_attention.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb" target="_blank">Colab</a> |
| #29 | Adam Optimizer | 🟡 Medium | `zeros_like` optimizer state and in-place parameter updates | [Open](29_adam.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/29_adam.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb" target="_blank">Colab</a> |
| #36 | INT8 Quantization | 🔴 Hard | `quantize_int8`: round, clamp, then cast, in that order | [Open](36_int8_quantization.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/duoan/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.